In [ ]:
# Manipulação de dados
import numpy as np
import pandas as pd

# Visualização de dados
import seaborn as sns
import matplotlib.pyplot as plt

# Construção do modelo
import tensorflow as tf

# Pré-processamento dos dados
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Funções auxiliares do modelo
from AirMLP_Model import sample_hyperparameters, createMLP_andTrain

## Funções

In [ ]:
def createWindows(df, windowSize):

    # Separa as variáveis de entrada e a variável alvo
    X = df.drop(columns=['valid_at', 'valore_originale', 'pm2p5_y']).values
    y = df['pm2p5_y'].values

    X_sep = []
    y_sep = []

    # Cria janelas deslizantes nos dados
    for i in range(len(X) - windowSize + 1):

        # Obtém e achata a janela de características
        janela_features = X[i : i + windowSize].flatten()
        X_sep.append(janela_features)

        # Define como alvo o último valor da janela
        alvo = y[i + windowSize - 1]
        y_sep.append(alvo)

    # Converte as listas em DataFrames
    X_final = pd.DataFrame(np.array(X_sep))
    y_final = pd.DataFrame(np.array(y_sep))

    return X_final, y_final

In [ ]:
def createModelData(Data_Mar_01, Data_Mar_02, Data_Mar_03, Data_Oct_01, Data_Oct_02, windowSize, norm=False):
    # Cria as janelas para cada conjunto de dados
    X_Mar_01, y_Mar_01 = createWindows(Data_Mar_01, windowSize)
    X_Mar_02, y_Mar_02 = createWindows(Data_Mar_02, windowSize)
    X_Mar_03, y_Mar_03 = createWindows(Data_Mar_03, windowSize)
    X_Oct_01, y_Oct_01 = createWindows(Data_Oct_01, windowSize)
    X_Oct_02, y_Oct_02 = createWindows(Data_Oct_02, windowSize)

    # Combina todos os conjuntos em um único dataset
    X = pd.concat([X_Mar_01, X_Mar_02, X_Mar_03, X_Oct_01, X_Oct_02], ignore_index=True)
    y = pd.concat([y_Mar_01, y_Mar_02, y_Mar_03, y_Oct_01, y_Oct_02], ignore_index=True)

    # Divide os dados em treino, validação e teste
    X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size=0.75, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, train_size=0.50, random_state=42)

    # Reordena os dados
    X_train.sort_index(inplace=True)
    X_val.sort_index(inplace=True)
    X_test.sort_index(inplace=True)
    X_aux.sort_index(inplace=True)

    y_train.sort_index(inplace=True)
    y_val.sort_index(inplace=True)
    y_test.sort_index(inplace=True)
    y_aux.sort_index(inplace=True)

    # Normaliza os dados, se solicitado
    if norm:
        scaler_X = StandardScaler()
        X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val = pd.DataFrame(scaler_X.transform(X_val), columns=X_val.columns, index=X_val.index)
        X_test = pd.DataFrame(scaler_X.transform(X_test), columns=X_test.columns, index=X_test.index)

        scaler_y = StandardScaler()
        y_train = pd.DataFrame(scaler_y.fit_transform(y_train.values.reshape(-1, 1)), columns=['target'], index=y_train.index)
        y_val = pd.DataFrame(scaler_y.transform(y_val.values.reshape(-1, 1)), columns=['target'], index=y_val.index)
        y_test = pd.DataFrame(scaler_y.transform(y_test.values.reshape(-1, 1)), columns=['target'], index=y_test.index)

    return X_train, y_train, X_val, y_val, X_test, y_test

## Dados

In [ ]:
# Carrega os dados
ari_Mar_01 = pd.read_csv('Dados/ari-1727.csv')[::-1].reset_index(drop=True)
ari_Mar_02 = pd.read_csv('Dados/ari-1952.csv')[::-1].reset_index(drop=True)
ari_Mar_03 = pd.read_csv('Dados/ari-1953.csv')[::-1].reset_index(drop=True)
ari_Oct_01 = pd.read_csv('Dados/ari-1885.csv')[::-1].reset_index(drop=True)
ari_Oct_02 = pd.read_csv('Dados/ari-2049.csv')[::-1].reset_index(drop=True)
rf_01      = pd.read_csv('Dados/rf_1.csv')
rf_02      = pd.read_csv('Dados/rf_2.csv')

# Remove valores ausentes
ari_Mar_01.dropna(inplace=True)
ari_Mar_02.dropna(inplace=True)
ari_Mar_03.dropna(inplace=True)
ari_Oct_01.dropna(inplace=True)
ari_Oct_02.dropna(inplace=True)
rf_01.dropna(inplace=True)
rf_02.dropna(inplace=True)

# Converte a coluna de data para datetime
ari_Mar_01['valid_at'] = pd.to_datetime(ari_Mar_01['valid_at'])
ari_Mar_02['valid_at'] = pd.to_datetime(ari_Mar_02['valid_at'])
ari_Mar_03['valid_at'] = pd.to_datetime(ari_Mar_03['valid_at'])
ari_Oct_01['valid_at'] = pd.to_datetime(ari_Oct_01['valid_at'])
ari_Oct_02['valid_at'] = pd.to_datetime(ari_Oct_02['valid_at'])
rf_01['valid_at'] = pd.to_datetime(rf_01['valid_at'])
rf_02['valid_at'] = pd.to_datetime(rf_02['valid_at'])

# Ajusta os dados para o mesmo fuso horário
rf_01['valid_at'] = rf_01['valid_at'] - pd.Timedelta(hours=1)
rf_02['valid_at'] = rf_02['valid_at'] - pd.Timedelta(hours=1)

# Arredonda os horários para intervalos de uma hora
ari_Mar_01["valid_at"] = ari_Mar_01["valid_at"].dt.round("h")
ari_Mar_02["valid_at"] = ari_Mar_02["valid_at"].dt.round("h")
ari_Mar_03["valid_at"] = ari_Mar_03["valid_at"].dt.round("h")
ari_Oct_01["valid_at"] = ari_Oct_01["valid_at"].dt.round("h")
ari_Oct_02["valid_at"] = ari_Oct_02["valid_at"].dt.round("h")
rf_01["valid_at"] = rf_01["valid_at"].dt.round("h")
rf_02["valid_at"] = rf_02["valid_at"].dt.round("h")

# Combina os dados das estações ARI e RF
Data_Mar_01 = pd.merge(ari_Mar_01, rf_01, on='valid_at', how='inner')
Data_Mar_02 = pd.merge(ari_Mar_02, rf_01, on='valid_at', how='inner')
Data_Mar_03 = pd.merge(ari_Mar_03, rf_01, on='valid_at', how='inner')
Data_Oct_01 = pd.merge(ari_Oct_01, rf_02, on='valid_at', how='inner')
Data_Oct_02 = pd.merge(ari_Oct_02, rf_02, on='valid_at', how='inner')

### Seed aleatória

In [ ]:
# Define a semente para reprodutibilidade dos resultados
SEED = 42
random.seed(SEED)        # Python
np.random.seed(SEED)     # NumPy
tf.random.set_seed(SEED) # TensorFlow

## Otimização Etapa 01

In [ ]:
# Realiza a busca aleatória de hiperparâmetros
Opt_exp = None

for _ in range(400):
    params = sample_hyperparameters(
        winSize_list=[10, 20, 30],
        hiddenLayer_range=(4, 10),
        neuron_list=[64, 128, 256, 512],
        function_list=['relu', 'tanh', 'sigmoid'],
        learning_rate_range=(-5, -2),
        batch_size_list=[64, 128, 256, 512]
    )

    # Prepara os dados para o treinamento
    X_train, y_train, X_val, y_val, X_test, y_test = createModelData(
        Data_Mar_01, Data_Mar_02, Data_Mar_03,
        Data_Oct_01, Data_Oct_02,
        params['winSize'],
        norm=True
    )

    # Treina o modelo com os hiperparâmetros amostrados
    result = createMLP_andTrain(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        hiddenLayers=params['hiddenLayers'],
        neurons=params['neurons'],
        function=params['function'],
        learning_rate=params['learning_rate'],
        batchSize=params['batchSize'],
        epochs=params['epochs']
    )

    # Armazena os resultados do experimento
    if Opt_exp is None:
        Opt_exp = pd.DataFrame([params | result])
    else:
        Opt_exp = pd.concat([Opt_exp, pd.DataFrame([params | result])], ignore_index=True)

    # Salva os resultados em arquivo
    Opt_exp.to_csv('Opt_exp_01.csv', index=False)

## Otimização Etapa 02

In [ ]:
# Realiza uma busca refinada de hiperparâmetros
Opt_exp = None

for _ in range(400):
    params = sample_hyperparameters(
        winSize_list=[20, 30],          # Janelas mais longas
        hiddenLayer_range=(4, 7),       # Redes mais rasas
        neuron_list=[256, 512, 1024],   # Número maiores de neurônios
        function_list=['relu', 'tanh'], # Funções de ativação selecionadas
        learning_rate_range=(-4, -2.5), # Taxas de aprendizado mais promissoras
        batch_size_list=[64, 128]       # Menores valores de batch size
    )

    # Prepara os dados para o treinamento
    X_train, y_train, X_val, y_val, X_test, y_test = createModelData(
        Data_Mar_01, Data_Mar_02, Data_Mar_03,
        Data_Oct_01, Data_Oct_02,
        params['winSize'],
        norm=True
    )

    # Treina o modelo com os hiperparâmetros amostrados
    result = createMLP_andTrain(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        hiddenLayers=params['hiddenLayers'],
        neurons=params['neurons'],
        function=params['function'],
        learning_rate=params['learning_rate'],
        batchSize=params['batchSize'],
        epochs=params['epochs']
    )

    # Armazena e salva os resultados dos experimentos
    if Opt_exp is None:
        Opt_exp = pd.DataFrame([params | result])
    else:
        Opt_exp = pd.concat([Opt_exp, pd.DataFrame([params | result])], ignore_index=True)

    # Salva os resultados em arquivo
    Opt_exp.to_csv('Opt_exp_02.csv', index=False)

72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
72/72 ━━━━━━━━━━━━━━━━━━━